In [0]:
import json, os, time, random, uuid
from datetime import datetime, timezone

RAW = "/Volumes/streaming/landing/inbound/raw/"
os.makedirs(RAW, exist_ok=True)

# ── realistic reference data ───────────────────────────────
WIKIS       = ["enwiki","dewiki","frwiki","eswiki","jawiki","arwiki","zhwiki","ptwiki"]
NAMESPACES  = [0, 0, 0, 1, 2, 4, 10]
EVENT_TYPES = ["edit","edit","edit","new","log"]
BOT_USERS   = [f"Bot{i}" for i in range(1, 12)]
HUMAN_USERS = [f"User_{uuid.uuid4().hex[:6]}" for _ in range(80)]
USERS       = HUMAN_USERS + BOT_USERS
TITLES = [
    "Python_(programming_language)","Quantum_mechanics","Chennai",
    "Albert_Einstein","World_War_II","Machine_learning","Cricket",
    "India","Solar_system","Climate_change","SpaceX","Netflix",
    "Artificial_intelligence","Linux","HTTP","Delta_Lake","Apache_Spark",
]

batch_size = 50
file_count = 0
total      = 0

print(f"✓ Raw folder      : {RAW}")
print(f"✓ Generator started — writing {batch_size} events per file every 10s\n")

start_time = time.time()
duration = 1 * 60  # 5 minutes in seconds

while time.time() - start_time < duration:
    records = []
    now     = int(datetime.now(timezone.utc).timestamp())

    for _ in range(batch_size):
        user    = random.choice(USERS)
        is_bot  = user in BOT_USERS
        len_old = random.randint(100, 50000)
        len_new = len_old + random.randint(-500, 2000)

        rec = {
            "id":          random.randint(10**9, 10**10),
            "type":        random.choice(EVENT_TYPES),
            "title":       random.choice(TITLES),
            "user":        user,
            "bot":         is_bot,
            "timestamp":   now - random.randint(0, 30),
            "server_name": "en.wikipedia.org",
            "wiki":        random.choice(WIKIS),
            "namespace":   random.choice(NAMESPACES),
            "meta": {
                "domain": "en.wikipedia.org",
                "dt":     datetime.utcnow().isoformat() + "Z",
                "id":     str(uuid.uuid4()),
            },
            "length": {
                "old": len_old,
                "new": max(0, len_new),
            },
        }
        records.append(rec)

    fname = f"{RAW}wiki_{now}_{file_count}.ndjson"
    with open(fname, "w") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")

    total      += len(records)
    file_count += 1
    print(f"[{datetime.utcnow().strftime('%H:%M:%S')}]  "
          f"file {file_count:>4}  |  {len(records)} events  |  "
          f"total {total:>6}  |  wiki_{now}_{file_count}.ndjson")

    time.sleep(10)